In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [ ]:
def processPdf(path):
    documents = [];
    directory = Path(path)

    pdfFiles = list(directory.glob("**/*.pdf"))
    print(len(pdfFiles))

    # print("length: ",len(pdfFiles))

    for pdf in pdfFiles: 
        # print(pdf.name)

        try:
            loader = PyMuPDFLoader(str(pdf));
            doc = loader.load()          
            documents.extend(doc)
        except:
            pass
    return documents

docs = processPdf("../data");
# print(docs)

In [ ]:
## test splitting
def createChunks(docs, chunkSize=1000,chunkOverlap=200):
    textSplitter = RecursiveCharacterTextSplitter(
        chunk_size=chunkSize,
        chunk_overlap= chunkOverlap,
        separators=['\n\n', '\n', ' ', ''],
        length_function=len
    )

    splitDocs = textSplitter.split_documents(docs)
    return splitDocs

chunks = createChunks(docs)
chunks

In [ ]:
# embedding

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

In [ ]:
class EmbeddingManager:
    def __init__(self, modelName = "all-MiniLM-L6-v2"):
        self.modelName = modelName
        self.model = None
        self._loadModel()
    # _ in start: protected function in python classes, only accessable in this class
    def _loadModel(self):
        try:
            self.model = SentenceTransformer(self.modelName)
            # print("model dimensions",self.model.get_sentence_embedding_dimension())
        except Exception as e:
            print("error: ",e);
            raise


    def generateEmbedding(self,text:List[str])->np.ndarray:
        if not self.model:
            print("model not found")
            return;

        print("text length", len(text))
        embedding = self.model.encode(text,show_progress_bar=True)
        print(f"embedding shape {embedding.shape}")
        return embedding;

    def getEmbeddingDimensions(self) -> int:
        if not self.model:
            print("model not loaded");
        return self.model.get_sentence_embedding_dimension()


In [ ]:
embeddingInstant = EmbeddingManager()
embeddingInstant.generateEmbedding([doc.page_content for doc in chunks])

In [ ]:
class VectorStore:
    def __init__(self, collectionName = "pdfDocs", persistDir = "../data/vectorStore"):
        self.collectionName = collectionName
        self.persistDir = persistDir
        self.client = None
        self.collection = None
        self._initializeStore_()

    def _initializeStore_(self):
        try:
            os.makedirs(self.persistDir, exist_ok = True)
            self.client = chromadb.PersistentClient(path = self.collectionName)        
            self.collection =self.client.get_or_create_collection(name=self.collectionName,metadata={"description": "PDF for RAG"})

            print("vector store initialize")
        except Exception as e:
            print(e)

    def addDocs(self, docs:List[Any], embedding:np.ndarray):
        ids = []
        metaDatas = []
        docText= []
        embeddingList = []

        for i, (doc, embedding) in enumerate(zip(docs, embedding)):
            docId = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(docId)

            metaData = dict(doc.metadata)
            metaData['doc_index']= i
            metaData["content_length"] = len(doc.page_content)

            metaDatas.append(metaData)
            docText.append(doc.page_content)
            embeddingList.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings = embeddingList,
                metadatas=metaDatas,
                documents = docText
            )


        except Exception as e:
            print("error adding docs to vector store: ", e)

vectorStoreInstance = VectorStore()


vector store initialize


In [ ]:
texts = [doc.page_content for doc in chunks]
embeddings = embeddingInstant.generateEmbedding(texts)
vectorS = vectorStoreInstance.addDocs(chunks, embeddings)
print(vectorS)


text length 152


Batches: 100%|██████████| 5/5 [00:05<00:00,  1.19s/it]


embedding shape (152, 384)
embeddingList [[0.03979417681694031, -0.004487964790314436, -0.058577168732881546, -0.0013491815188899636, 0.09271182864904404, 0.033328067511320114, -0.11200352013111115, 0.04928545281291008, -0.0040693567134439945, 0.09997118264436722, 0.00653991149738431, 0.0432039275765419, -0.05029201880097389, -0.028200220316648483, -0.036456987261772156, -0.007084690500050783, -0.02975630760192871, 0.08210893720388412, 0.04681866616010666, -0.011238991282880306, -0.025059513747692108, -0.005050674546509981, 0.03874450922012329, -0.006836744491010904, 0.023800892755389214, -0.013035744428634644, -0.03855479136109352, -0.03363678604364395, -0.043021511286497116, -0.06488581001758575, -0.054262351244688034, 0.042354051023721695, -0.0925046056509018, -0.031197864562273026, 0.05878502130508423, -0.006601524073630571, -0.030419472604990005, 0.03846493363380432, 0.0006969374953769147, -0.047648124396800995, 0.01601526513695717, -0.01886073686182499, -0.011697322130203247, 0.0

Take the user query and convert it to the embeddings and then hit the vector store in the form of retrival.

In [ ]:
class QueryRetrival:
    def __init__(self, vectorStore:VectorStore, embeddingManager:EmbeddingManager):
        self.vectorStore = vectorStore
        self.embeddingManager = embeddingManager

    def query(self, queryText:str, topK:int = 5,scoreThreshhold:float = 0.5) -> List[Dict[str,Any]]:
        # topK = number of top results to return which matches the query
        # scoreThreshhold = minimium similarity score to consider a match

        print("query text", queryText)
        print("topK", topK)
        print("scoreThreshhold", scoreThreshhold)

        queryEmbedding = self.embeddingManager.generateEmbedding([queryText])[0]

        # print("query embedding", queryEmbedding)


        try:
            result = self.vectorStore.collection.query(
                query_embeddings = [queryEmbedding],
                n_results = topK,
            )

            print("query result", result)
            retrievedDoc = []

            if(result["documents"] and result["documents"][0]):
                documents = result["documents"][0]
                metadatas = result["metadatas"][0]
                distances = result["distances"][0]
                ids = result["ids"][0]

                for i,(docId,doc, metadata, distance) in enumerate(zip(ids, documents,metadatas,distances)):
                    similarityScore = 1- distance

                    if similarityScore >= scoreThreshhold:
                        retrievedDoc.append({
                            "id":docId,
                            "metadata": metadata,
                            "doc": doc,
                            "distance": distance,
                            "similarityScore":similarityScore,
                            "rank": i+1
                        }) 
            else:
                print(f"No doc found.")
            return retrievedDoc 
        except Exception as e:
            print("error in QueryRetrival class: ",e)
            return []

In [61]:
queryRetrival = QueryRetrival(vectorStoreInstance, embeddingInstant)
queryRetrival.query("What am I doing here in this endless winter?", topK=3, scoreThreshhold=0.4)

query text What am I doing here in this endless winter?
topK 3
scoreThreshhold 0.4
text length 1


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.96it/s]

embedding shape (1, 384)
query result {'ids': [['doc_f7281722_38', 'doc_bf80200b_3', 'doc_73a0547e_26']], 'embeddings': None, 'documents': [['me to set out on my way, will you not? You see, Mr. Manag\xad\ner, I am not pig-headed, and I am happy to work. Traveling \nis exhausting, but I couldn’t live without it. Where are you \ngoing, Mr. Manager? To the office? Really? Will you report \neverything truthfully? A person can be incapable of work \nmomentarily, but that is precisely the best time to remem\xad\nber the earlier achievements and to consider that later, after \nthe obstacles have been shoved aside, the person will work \nall the more keenly and intensely. I am really so indebted to \nMr. Chief—you know that perfectly well. On the other hand, \nI am concerned about my parents and my sister. I’m in a fix, \nbut I’ll work myself out of it again. Don’t make things more \ndifficult for me than they already are. Speak up on my behalf \nin the office! People don’t like traveling sale

[]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
gemini_apikey = os.getenv("GEMINI_API_KEY")

llmModel = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=1.0,  # Gemini 3.0+ defaults to 1.0
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

def generateAns(query,retriver,llmModel,topK=3):
    result = retriver.query(query, topK = topK)
    content = "\n\n".join([f"Content: {doc['doc']}" for doc in result]) if result else "No relevant content found."
    if not content: 
        return "No relevant content found."
    
    prompt = f"Answer the question based on the following retrieved content:\n\n{content}\n\nQuestion: {query}\nAnswer:"

    response = llmModel.invoke([prompt.format(content=content, query=query)])
    return response.content

In [ ]:
ans = generateAns("I cannot make you understand. I cannot make anyone understand what is happening inside me. I cannot even explain it to myself.",queryRetrival , llmModel)
print("ans: ",ans)